In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.pipeline import Pipeline

## Task 3.1 — Features, split and baseline

In [3]:
df = pd.read_csv('appointments_clean.csv')
df['reminder_sent'] = df['reminder_sent'].map({'Yes': 1, 'No': 0})
features_to_use = ['age','distance_km','fee_inr','lead_days','prior_no_shows','reminder_sent','booking_channel','visit_type']

X_raw = df[features_to_use]
y = df['no_show']

X = pd.get_dummies(X_raw, columns=["booking_channel","visit_type"],drop_first=False)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]} rows, {X_train.shape[1]} columns")
print(f"Test set size: {X_test.shape[0]} rows")
no_shows_in_test = y_test.sum()
print(f"Number of no-show in test set: {no_shows_in_test}")

baseline_acc = (y_test == 0).mean()
print(f"Baseline accuracy (always predict 'Attended'): {baseline_acc:.4f}")

Training set size: 792 rows, 11 columns
Test set size: 198 rows
Number of no-show in test set: 49
Baseline accuracy (always predict 'Attended'): 0.7525


## Task 3.2 — Logistic regression

In [4]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

lr_model = LogisticRegression(max_iter = 1000)
lr_model.fit(X_train_s, y_train)

lr_preds = lr_model.predict(X_test_s)
lr_probs = lr_model.predict_proba(X_test_s)[:, 1]

print("\nLogistic Regression (Scaled)")
print(f"Accuracy: {accuracy_score(y_test, lr_preds):.3f}")
print(f"Precision: {precision_score(y_test, lr_preds):.3f}")
print(f"Recall: {recall_score(y_test, lr_preds):.3f}")
print(f"F1 Score: {f1_score(y_test, lr_preds):.3f}")
print(f"ROC-AUC Score: {roc_auc_score(y_test, lr_probs):.3f}")
print(f"Confusion Matrix:\n", confusion_matrix(y_test, lr_preds))

feature_names = X.columns
coefficients = lr_model.coef_[0]

print("\nCoefficients:")
for name, coef in zip(feature_names, coefficients):
    print(f"{name}: {coef:.4f}")

reminder_idx = list(feature_names).index('reminder_sent')
reminder_coef = coefficients[reminder_idx]
reminder_scale = scaler.scale_[reminder_idx]

exp_coef_scaled = np.exp(reminder_coef)
exp_coef_real = np.exp(reminder_coef / reminder_scale)

print(f"\nnp.exp of scaled coefficient: {exp_coef_scaled:.4f}")
print(f"Real No->Yes change (np.exp(coef / scale)): {exp_coef_real:.4f}")

print("""
 Conclusion: In this model, holding other features is constant, the odds of a no-show are about 69% lower (0.31 multiplier) for appointments with a reminder, but since data is observational, this is an association, not proof of cause.
 They differ because np.exp(coef) gives the odds change per one standard deviation of the scaled feature, while dividing by scaler.scale_ converts it to the real No (0) to Yes (1) jump.
 """)


Logistic Regression (Scaled)
Accuracy: 0.788
Precision: 0.630
Recall: 0.347
F1 Score: 0.447
ROC-AUC Score: 0.789
Confusion Matrix:
 [[139  10]
 [ 32  17]]

Coefficients:
age: -0.2539
distance_km: 0.2534
fee_inr: 0.4528
lead_days: 0.6145
prior_no_shows: 0.6496
reminder_sent: -0.5807
booking_channel_App: -0.0468
booking_channel_Phone: 0.2159
booking_channel_Walk-In: -0.1864
visit_type_Consultation: 0.0157
visit_type_Follow-up: -0.0157

np.exp of scaled coefficient: 0.5595
Real No->Yes change (np.exp(coef / scale)): 0.3094

 Conclusion: In this model, holding other features is constant, the odds of a no-show are about 69% lower (0.31 multiplier) for appointments with a reminder, but since data is observational, this is an association, not proof of cause.
 They differ because np.exp(coef) gives the odds change per one standard deviation of the scaled feature, while dividing by scaler.scale_ converts it to the real No (0) to Yes (1) jump.
 


## Task 3.3 — Decision tree and overfitting

In [5]:
print("\nDecision Tree")

# Full Depth
dt_full = DecisionTreeClassifier(random_state=42)
dt_full.fit(X_train, y_train)

print(f"Full Depth - Training Accuracy: {dt_full.score(X_train, y_train):.3f}")
print(f"Full Depth - Test Accuracy: {dt_full.score(X_test, y_test):.3f}")

# Depth-3 tree
dt_lim = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_lim.fit(X_train, y_train)

print(f"Depth 3 - Training Accuracy: {dt_lim.score(X_train, y_train):.3f}")
print(f"Depth 3 - Test Accuracy: {dt_lim.score(X_test, y_test):.3f}")

print("Conclusion: The full-depth tree achieves 100% training accuracy but performs worse than the baseline on the test set, showing it memorized noise (overfitting).")



Decision Tree
Full Depth - Training Accuracy: 1.000
Full Depth - Test Accuracy: 0.672
Depth 3 - Training Accuracy: 0.822
Depth 3 - Test Accuracy: 0.778
Conclusion: The full-depth tree achieves 100% training accuracy but performs worse than the baseline on the test set, showing it memorized noise (overfitting).


## Task 3.4 — Random forest and a comparison table

In [6]:
print("\nRandom Forest")

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_test)
rf_probs = rf_model.predict_proba(X_test)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, rf_preds):.3f}")
print(f"Recall: {recall_score(y_test, rf_preds):.3f}")
print(f"ROC-AUC : {roc_auc_score(y_test, rf_probs):.3f}")

importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]
print("\nTop 3 feature Importances:")
for i in range(3):
  print(f"{feature_names[indices[i]]}: {importances[indices[i]]:.3f}")

comp_data = {
    'Model': ['Logistic Regression', 'Decision Tree (Depth 3)', 'Random Forest'],
    'Accuracy': [accuracy_score(y_test, lr_preds), dt_lim.score(X_test, y_test), accuracy_score(y_test, rf_preds)],
    'Recall': [recall_score(y_test, lr_preds), recall_score(y_test, dt_lim.predict(X_test)), recall_score(y_test, rf_preds)],
    'ROC-AUC': [roc_auc_score(y_test, lr_probs), roc_auc_score(y_test, dt_lim.predict_proba(X_test)[:, 1]), roc_auc_score(y_test, rf_probs)]
}

comp_df = pd.DataFrame(comp_data).round(3)
print("\nComparison Table (Test Set):")
print(comp_df.to_string(index=False))

print("\nObservation: Logistic Regression looks best because it has the highest ROC-AUC and the best balance of Accuracy and Recall on the test set, but formal selection must rely on cross-validation.")


Random Forest
Accuracy: 0.747
Recall: 0.306
ROC-AUC : 0.706

Top 3 feature Importances:
lead_days: 0.220
age: 0.197
distance_km: 0.195

Comparison Table (Test Set):
                  Model  Accuracy  Recall  ROC-AUC
    Logistic Regression     0.788   0.347    0.789
Decision Tree (Depth 3)     0.778   0.224    0.700
          Random Forest     0.747   0.306    0.706

Observation: Logistic Regression looks best because it has the highest ROC-AUC and the best balance of Accuracy and Recall on the test set, but formal selection must rely on cross-validation.


## Task 3.5 — Tune, choose the model and the threshold, then test once

In [7]:
print("\nModel Tuning and Selection")

#Tune Random Forest
param_grid = {"n_estimators": [100, 200], "max_depth":[3,5,8]}
grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, scoring="roc_auc")
grid_search.fit(X_train, y_train)

print(f"GridSearchCV Best Params: {grid_search.best_params_}")
print(f"GridSearchCV Best Score: {grid_search.best_score_:.3f}")
tuned_rf = grid_search.best_estimator_

# Cross Validation
lr_pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])
cv_auc_lr = cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean()
cv_auc_dt = cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=42), X_train, y_train, cv=5, scoring="roc_auc").mean()
cv_auc_rf = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42), X_train, y_train, cv=5, scoring="roc_auc").mean()

print("\nCross-Validated Training ROC-AUC:")
print(f"Logistic Regression: {cv_auc_lr:.3f}")
print(f"Decision Tree (Depth 3): {cv_auc_dt:.3f}")
print(f"Random Forest: {cv_auc_rf:.3f}")
print(f"Tuned Random Forest: {grid_search.best_score_:.3f}")
print("-> Logistic Regression wins (Highest CV ROC-AUC).")

# Choose threshold using Out of Fold(OOF) predictions
print("\nThreshold selection (using Logistic Regression OOF Probs)")
oof_probs = cross_val_predict(lr_pipe, X_train, y_train, cv=5, method="predict_proba")[:, 1]

def evaluate_threshold(probs, thresh, y_true):
  preds = (probs >= thresh).astype(int)
  return {
      'Precision': precision_score(y_true, preds),
      'Recall': recall_score(y_true, preds),
      'F1': f1_score(y_true, preds),
      'Flagged Count': preds.sum()
  }

metrics_05 = evaluate_threshold(oof_probs, 0.5, y_train)
metrics_035 = evaluate_threshold(oof_probs, 0.35, y_train)

print(f"Threshold 0.5: Precision ={metrics_05['Precision']:.2f}, Recall = {metrics_05['Recall']:.2f}, F1 = {metrics_05['F1']:.2f}, Flagged = {metrics_05['Flagged Count']}")
print(f"Threshold 0.35: Precision={metrics_035['Precision']:.2f}, Recall={metrics_035['Recall']:.2f}, F1={metrics_035['F1']:.2f}, Flagged={metrics_035['Flagged Count']}")

print("Decision: Choose threshold 0.35 because it yields a higher F1 score and much higher recall. Recall matters here because a missed warning costs a completely empty slot, whereas an extra call only costs a minute of staff time. ")

#Final Evaluation on Test Set
print("\nFinal Test Set Evaluation (LR at Threshold 0.35)")
lr_pipe.fit(X_train, y_train)
final_test_probs = lr_pipe.predict_proba(X_test)[:, 1]
final_test_preds = (final_test_probs >= 0.35).astype(int)

final_prec = precision_score(y_test, final_test_preds)
final_rec = recall_score(y_test, final_test_preds)
final_flagged = final_test_preds.sum()
final_auc = roc_auc_score(y_test, final_test_probs)

print(f"Precision: {final_prec:.2f}")
print(f"Recall: {final_rec:.2f}")
print(f"Number Flagged: {final_flagged}")
print(f"ROC-AUC: {final_auc:.2f}")

#Save test_risk_scores.csv
id_test = df.loc[X_test.index, 'appointment_id']
results_df = pd.DataFrame({
    'appointment_id': id_test,
    'no_show': y_test,
    'risk_probs': final_test_probs,
    'flag': final_test_preds
})

results_df.to_csv('test_risk_scores.csv', index=False)
print("\ntest_risk_scores.csv saved successfully!")

print("\nReal World Application: A clinic would really choose the threshold by evaluating how many follow-up calls the front desk can realistically make in an evening, balanced against the financial cost of an empty slot.")


Model Tuning and Selection
GridSearchCV Best Params: {'max_depth': 3, 'n_estimators': 200}
GridSearchCV Best Score: 0.767

Cross-Validated Training ROC-AUC:
Logistic Regression: 0.774
Decision Tree (Depth 3): 0.678
Random Forest: 0.747
Tuned Random Forest: 0.767
-> Logistic Regression wins (Highest CV ROC-AUC).

Threshold selection (using Logistic Regression OOF Probs)
Threshold 0.5: Precision =0.58, Recall = 0.25, F1 = 0.35, Flagged = 73
Threshold 0.35: Precision=0.51, Recall=0.47, F1=0.49, Flagged=154
Decision: Choose threshold 0.35 because it yields a higher F1 score and much higher recall. Recall matters here because a missed warning costs a completely empty slot, whereas an extra call only costs a minute of staff time. 

Final Test Set Evaluation (LR at Threshold 0.35)
Precision: 0.57
Recall: 0.57
Number Flagged: 49
ROC-AUC: 0.79

test_risk_scores.csv saved successfully!

Real World Application: A clinic would really choose the threshold by evaluating how many follow-up calls the